# Week 1, Lab 1 — Intro & First Call to a Local Model

**Course:** Agentic AI Engineering — Local Models Edition
**Author:** Abhishek

Runs identically on **Google Colab** (Hugging Face `transformers`) and on
**your own PC** (Ollama) — no API key, no cost, either way.

## What you'll do
1. Detect whether you're on Colab or local, and pick the right backend
2. Make your first call to a local LLM
3. Try zero-shot vs. few-shot vs. chain-of-thought prompting


## 1. Environment detection

Run this cell as-is. It figures out where you are and tells you what to do next.


In [35]:
import importlib.util


BACKEND = "ollama"

print(f"Detected environment: 'Local PC'")
print(f"Backend selected: {BACKEND}")


print("\nMake sure Ollama is running: `ollama serve` in a terminal, and you've")
print("pulled a model: `ollama pull llama3.2:1b`")


Detected environment: 'Local PC'
Backend selected: ollama

Make sure Ollama is running: `ollama serve` in a terminal, and you've
pulled a model: `ollama pull llama3.2:1b`


## 2. Install/import what this backend needs

- **Colab (huggingface):** installs `transformers`, `torch`, `accelerate`
- **Local (ollama):** just needs the `ollama` python package (talks to your
  already-running local Ollama server)


In [36]:
#  install ollalm


In [37]:
import warnings
warnings.filterwarnings("ignore")

if BACKEND == "huggingface":
    from transformers import pipeline
    import torch

    HF_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # swap for Qwen2.5-0.5B-Instruct if slow / CPU-only
    device = 0 if torch.cuda.is_available() else -1
    print(f"Loading {HF_MODEL} on {'GPU' if device == 0 else 'CPU'} ...")
    generator = pipeline("text-generation", model=HF_MODEL, device=device)

    def local_chat(messages, max_new_tokens=256, temperature=0.7):
        """messages: list of {"role": "system"|"user"|"assistant", "content": str}"""
        output = generator(
            messages,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
        )
        return output[0]["generated_text"][-1]["content"]

else:
    import ollama

    OLLAMA_MODEL = "llama3.2:1b"

    def local_chat(messages, max_new_tokens=256, temperature=0.7):
        response = ollama.chat(
            model=OLLAMA_MODEL,
            messages=messages,
            options={"num_predict": max_new_tokens, "temperature": temperature},
        )
        return response["message"]["content"]

print("local_chat() is ready.")


local_chat() is ready.


## 3. First call

Same `local_chat()` function works no matter which backend was selected above.


In [38]:
response = local_chat([
    {"role": "system", "content": "You are a concise, helpful teaching assistant."},
    {"role": "user", "content": "In one sentence, what is an LLM agent?"},
])
print(response)


An LLM (Large Language Model) agent is a computer system that uses artificial intelligence and natural language processing capabilities to generate human-like text or respond to questions and requests in a specific context or domain.


## 4. Prompting patterns

### Zero-shot


In [39]:
print(local_chat([
    {"role": "user", "content": "Classify the sentiment as positive, negative, or neutral: "
                                 "'The course was confusing but the instructor was helpful.'"}
]))


I would classify the sentiment as neutral. The word "confusing" is used to describe the experience of taking the course, which implies that it may have had some drawbacks, but it's not a strongly negative statement. The addition of "but the instructor was helpful" suggests that despite the challenges, the instructor provided assistance and support, which counters any potential criticism of the course itself.


### Few-shot

In [40]:
few_shot_prompt = """Classify the sentiment as positive, negative, or neutral.

Text: "I loved this!"
Sentiment: positive

Text: "This was a waste of time."
Sentiment: negative

Text: "It was fine, nothing special."
Sentiment: neutral

Text: "The course was confusing but the instructor was helpful."
Sentiment:"""

print(local_chat([{"role": "user", "content": few_shot_prompt}], max_new_tokens=50,temperature=0.0))


Based on the text, I would classify the sentiment as positive. The words used in each text convey a sense of approval or satisfaction with the experience or situation being described. For example, "I loved this!" and "It was fine, nothing special


### Chain-of-thought

In [41]:
cot_prompt = (
    "A store has 120 apples. It sells 45% of them in the morning and 30 more "
    "in the afternoon. How many apples are left? Think step by step, then give "
    "the final answer on its own line as 'Answer: <number>'."
)
print(local_chat([{"role": "user", "content": cot_prompt}], max_new_tokens=200,temperature=0))


To find out how many apples are left, we need to calculate the total number of apples sold.

First, let's calculate the number of apples sold in the morning:
45% of 120 apples = 0.45 x 120 = 54 apples

Then, add the additional apples sold in the afternoon:
54 + 30 = 84 apples

Now, subtract the total number of apples sold from the original amount:
120 - 84 = 36 apples

Answer: <36>


## 5. Exercise

1. Re-run the sentiment classification few-shot prompt, but change the
   examples to a domain you care about (e.g. classifying support tickets by
   urgency instead of sentiment).
2. Try the same chain-of-thought math question with `temperature=0` vs
   `temperature=1.0` a few times — how much does the answer vary?
3. **(Colab users)** swap `HF_MODEL` for `Qwen/Qwen2.5-0.5B-Instruct` and
   compare speed and quality.
   **(Local users)** swap `OLLAMA_MODEL` for `qwen2.5:3b` and compare.

## Where this goes next
`lab2_structured_output.ipynb` — getting the model to reliably return JSON
instead of free text, which every later week's agent frameworks depend on.
